### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following
- tracking agent behaviour with logging, analytics, and debugging.
- transforming prompts, tool selection, and output formatting.
- adding retries, fallbacks, and early termination logic.
- applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model
groq_model = init_chat_model("groq:qwen/qwen3-32b")
gemini_model = init_chat_model("google_genai:gemini-2.5-flash-lite")

#### Summarization Middleware
automatically summarize the conversation history when approaching token limits, preserving recent messages while compressing older context (something like compacting context in claude code).

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, SystemMessage

# message based summarization
agent = create_agent(
    model=groq_model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = groq_model,
            trigger = ("messages", 10),
            keep = ("messages", 4)
        )
    ]
)

In [7]:
# run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [8]:
questions = [
    "what is 23+45 ?",
    "what is 23-12 ?",
    "what is 45*212 ?",
    "what is 22341*65 ?",
    "what is 523/32 ?",
    "what is 975-62 ?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 23+45 ?', additional_kwargs={}, response_metadata={}, id='2ac1332b-0636-4bfe-a14d-ce1f1df05989'), AIMessage(content="<think>\nOkay, so I need to figure out what 23 plus 45 is. Let me start by breaking it down. Maybe I can add the tens and the ones separately. \n\nFirst, let's look at the tens place. 23 has two tens, which is 20, and 45 has four tens, which is 40. If I add those together, 20 plus 40 equals 60. \n\nNow the ones place. 23 has three ones, and 45 has five ones. Adding those gives 3 + 5, which is 8. \n\nSo if I combine the tens and the ones, it's 60 + 8. Let me check that again. 60 plus 8 is 68. Hmm, that seems right. \n\nWait, maybe I can do it another way to confirm. Let's try adding vertically. \n\n  23\n+45\n----\nStarting from the rightmost digit: 3 + 5 = 8. Then the tens place: 2 + 4 = 6. So putting them together, it's 68. \n\nAnother way could be to count up from 23 by 45. Starting at 23, adding 40 would be 63, the

#### token size

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.tools import tool
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $150/night, business center
    3. Budget Stay - 3 star, $80/night, free wifi"""
    
agent = create_agent(
    model=groq_model,
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model=groq_model,
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # taking 4 chars as 1 token

In [16]:
# run test
cities = ["paris", "london", "tokyo", "dubai", "singapore", "dharmshala"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

paris: ~164 tokens, 4 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='23ce75b6-c0b1-46ec-ae96-6ea1d75480c0'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to find hotels in Paris. Let me check the available tools. There's a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. Since the user specified Paris, I should use that as the argument. The function returns a long response, so I should generate a detailed answer. Let me structure the tool call properly.\n", 'tool_calls': [{'id': 'kg1mp9zjb', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 155, 'total_tokens': 266, 'completion_time': 0.228096313, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.017654915, 'promp

#### Fraction

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """search hotels"""
    return f"Hotels in {city}: Grand hotel $350, City Inn $150, Budget Stay $80"

agent = create_agent(
    model=groq_model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=groq_model,
            trigger=("fraction", 0.005), # 0.5% = ~640 tokens
            keep=("fraction", 0.002)     # 0.2% = ~256 tokens
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # taking 4 chars as 1 token


# run test
cities = ["paris", "london", "tokyo", "dubai", "singapore", "dharmshala"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    fraction = tokens/128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

paris: ~72 tokens (0.0562%), 4 messages
[HumanMessage(content='hotels in paris', additional_kwargs={}, response_metadata={}, id='23e4d9b6-c9f9-4c7e-b1bf-dd599a7ca329'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that requires a city parameter. Since the user mentioned Paris, I need to call that function with "Paris" as the city. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': '2f8dqph15', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 147, 'total_tokens': 244, 'completion_time': 0.150016327, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.041407279, 'prompt_tokens_details': None, 'queue_time': 0.528859994, 'total_

### Human In the Loop Middleware

Pause agent execution for human approval, editing, rejection of tool calls before they execute. Human in the loop is useful for the following

- High stakes operations requiring human approval
- Workflows where human oversight is mandatory.
- Long running conversation where human feedback guides the agent.

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock  function to read email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [29]:
agent = create_agent(
    model=groq_model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "rejection"]
                },
                "read_email_tool": False
            }
        )
    ]
)

In [30]:
config = {"configurable": {"thread_id": "test-approve"}}

# step1: send request
result = agent.invoke(
    {"messages": [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b91a07ef-d7ed-4fc9-8ef9-fb17984057fc'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are provided here, so I should call that function with the given parameters. No need to use the read_email_tool since the user isn't asking to read an email. Just need to structure the JSON correctly for the send_email_tool.\n", 'tool_calls': [{'id': '6ar6gr4s7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 251, 'total_tokens': 391,

In [31]:
from langgraph.types import Command

# step 2: approve
if "__interrupt__" in result:
    print("Paused! Approving ...")
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Approving ...
Result: The email has been successfully sent to john@test.com with the subject "Hello" and the body "How are you?". Let me know if you need anything else!


In [32]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b91a07ef-d7ed-4fc9-8ef9-fb17984057fc'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are provided here, so I should call that function with the given parameters. No need to use the read_email_tool since the user isn't asking to read an email. Just need to structure the JSON correctly for the send_email_tool.\n", 'tool_calls': [{'id': '6ar6gr4s7', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 251, 'total_tokens': 391,

##### Reject

In [47]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_emai_tool(email_id: str) -> str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model=groq_model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)

In [48]:
config = {"configurable": {"thread_id": "test-approve"}}

# step1: send request
result = agent.invoke(
    {"messages": [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [49]:
# step 2: reject
if "__interrupt__" in result:
    print("Paused! Approving ...")
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Approving ...
Result: Your request to send an email to **john@test.com** with the subject **"Hello"** was not processed. Please let me know if you'd like to proceed with sending it or if there's anything else I can assist you with.


In [50]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0d0b99af-ab9e-460b-9ea1-d39b86e8e114'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, the subject is 'Hello', and the body is 'How are you?'. I need to make sure all required fields are included. Yes, they are. So I should call the send_email_tool with these arguments. No other tools are needed here since it's a straightforward email sending request. I'll format the tool call accordingly.\n", 'tool_calls': [{'id': '7jmhd28w3', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'sen

##### Editing

In [51]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_emai_tool(email_id: str) -> str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model=groq_model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)

In [52]:
config = {"configurable": {"thread_id": "test-approve"}}

# step1: send request
result = agent.invoke(
    {"messages": [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [53]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1d4159b6-83c3-447c-8f9a-0c9350d7f0e6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. I need to make sure the arguments are correctly formatted. The recipient is john@test.com, subject is 'Hello', body is 'How are you?'. I'll structure the JSON with those values. No issues here, all required fields are provided. Let me call the send_email_tool with these parameters.\n", 'tool_calls': [{'id': 'efymhbfch', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}

In [55]:
# step 2: ediT and approve
if "__interrupt__" in result:
    print("Paused! Approving ...")
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Approving ...
Result: The email was sent to **correct@email.com** with the subject **"Corrected Subject"** instead of the original recipient **john@test.com** and subject **"Hello"**. 

Would you like me to:
1. Confirm the email details and resend to the intended recipient?
2. Use the `read_email_tool` to check the email content (if needed)? 

Let me know how you'd like to proceed!


In [56]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1d4159b6-83c3-447c-8f9a-0c9350d7f0e6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. I need to make sure the arguments are correctly formatted. The recipient is john@test.com, subject is 'Hello', body is 'How are you?'. I'll structure the JSON with those values. No issues here, all required fields are provided. Let me call the send_email_tool with these parameters.\n", 'tool_calls': [{'id': 'efymhbfch', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}